# DPO & REINFORCE, step by step — *watch the log-probabilities move*

*Hands-on companion to lecture sections 6 (Policy Gradient → PPO → GRPO) and 7 (DPO & Best-of-N).*
*Runs on a free Colab CPU or T4 — the model is tiny on purpose.*

**SOLUTION VERSION** — the three `TODO`s are filled in.


We make preference optimization **mechanically transparent**. To do that we use:

- a **toy reward model** — *be enthusiastic!* (counts `!` and upbeat words), so the target is obvious;
- a **tiny model** (`SmolLM2-135M-Instruct`), so we can look at the actual numbers and even train.

The notebook has **two complete, parallel parts** so you can compare the methods directly:

| | builds on | you implement |
|---|---|---|
| **Shared** | reward model · log-probability primitive `response_logps` ★ · eval helpers | `response_logps` |
| **Part A — DPO** | recap → one step → **full training** → distribution shift | `dpo_loss` ★ |
| **Part B — REINFORCE** | recap → one step → **full training** → distribution shift | `reinforce_loss` ★ |
| **Part C** | DPO vs REINFORCE, side by side | — |

> 💡 Optional but smoother: `Runtime → Change runtime type → T4 GPU`. CPU also works (sampling is slower).

## Setup

> 💡 **Before you start:** run the install cell below (≈1 min on Colab), then the model-loading cell.

In [ ]:
# Tiny install. transformers for the model; jinja2>=3.1 is needed by the chat template.
!pip -q install "transformers==4.57.1" "jinja2>=3.1.0" 2>/dev/null

import re, gc, random
import numpy as np
import torch
from torch.nn.functional import logsigmoid, log_softmax   # explicit names, no aliases
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

random.seed(0); np.random.seed(0); torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

### Load the model — *policy* vs *reference*

We load one tokenizer and two **roles** of the same tiny model: a trainable **policy** $\pi_\theta$
(made fresh with `load_model()` for each training run) and a frozen **reference** $\pi_{\mathrm{ref}}$ —
our "before" snapshot, and the KL anchor that DPO compares against.

In [ ]:
MODEL_NAME = "HuggingFaceTB/SmolLM2-135M-Instruct"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def load_model():
    """Fresh copy of the pretrained model on the right device, in float32 for clean gradients."""
    return AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=torch.float32).to(device)

# The REFERENCE model is frozen. It plays two roles:
#   * the "before training" policy we compare against, and
#   * the KL anchor inside the DPO loss (pi_ref).
reference_model = load_model().eval()
for parameter in reference_model.parameters():
    parameter.requires_grad_(False)

num_parameters = sum(parameter.numel() for parameter in reference_model.parameters())
print(f"loaded {MODEL_NAME}  ({num_parameters/1e6:.0f}M parameters)")

---
## Shared — A dummy reward model

In real RLHF the reward model is a neural network trained on human preferences (lecture section 5).
Here we swap it for a **3-line Python function** so the reward is completely transparent and we can
concentrate on the *optimizers*. Our toy preference is **enthusiasm**: reward exclamation marks and
upbeat words. It also returns a **breakdown** (which characters/words earned points) — used later to
*explain* each score.

In [ ]:
POSITIVE_WORDS = {
    "great", "love", "awesome", "amazing", "wonderful", "fantastic", "excited", "happy",
    "glad", "best", "incredible", "brilliant", "perfect", "excellent", "fun", "enjoy",
    "delightful", "joy", "beautiful", "lovely",
}

def reward_breakdown(response):
    """Return (reward, reasons). reward = (#exclamation marks) + (#positive words)."""
    lowercase = response.lower()
    exclamation_count = response.count("!")
    matched_words = []
    for word in POSITIVE_WORDS:
        occurrences = len(re.findall(r"\b" + word + r"\b", lowercase))
        matched_words.extend([word] * occurrences)
    reward = float(exclamation_count + len(matched_words))
    return reward, {"exclamation_marks": exclamation_count, "positive_words": matched_words}

def reward_function(response):
    """Scalar reward only (what the optimizers consume)."""
    reward, _ = reward_breakdown(response)
    return reward

for example_text in [
    "The weather is mild today.",
    "I love this, it is absolutely amazing!",
    "Wow!! What a fantastic, wonderful day!!",
]:
    reward, reasons = reward_breakdown(example_text)
    print(f"reward={reward:.0f}  | {example_text}")
    print(f"        reasons: {reasons['exclamation_marks']} '!'  +  words {reasons['positive_words']}")

Each `!` and each positive word is worth one point, and the breakdown shows *which* tokens scored — we
reuse that at the very end to explain the model's answers. This toy reward stands in for a trained
reward model, so nothing about the optimizers stays hidden.

---
## Shared ★ — The model's log-probability of a response

Both algorithms are built on **one** primitive: the log-probability a model assigns to a response,

$$\log \pi(y\mid x) \;=\; \sum_{t} \log \pi\!\left(y_t \mid x, y_{<t}\right).$$

We obtain every token's log-probability in **one forward pass** (teacher forcing): the logits at
position $t$ predict the token at position $t+1$. Your job is the gather-and-sum.

#### Generation & prompt helpers

`chat_prompt_text` wraps a message in the model's chat template; `sample_response` generates one
completion from any model. We also fix two prompt pools: `TRAINING_PROMPTS` (to build data and train
on) and `HELD_OUT_PROMPTS` (kept aside, to check the behavior *generalizes*).

In [ ]:
def chat_prompt_text(prompt):
    """Apply the chat template and add the assistant generation prompt."""
    return tokenizer.apply_chat_template(
        [{"role": "user", "content": prompt}],
        tokenize=False, add_generation_prompt=True,
    )

@torch.no_grad()
def sample_response(model, prompt, max_new_tokens=40, temperature=1.0):
    """Sample one completion from `model` for `prompt`."""
    input_ids = tokenizer(chat_prompt_text(prompt), return_tensors="pt",
                          add_special_tokens=False).input_ids.to(device)
    generated = model.generate(
        input_ids, attention_mask=torch.ones_like(input_ids),
        max_new_tokens=max_new_tokens, do_sample=temperature > 0,
        temperature=max(temperature, 1e-5), top_p=0.95,
        pad_token_id=tokenizer.eos_token_id,
    )
    return tokenizer.decode(generated[0, input_ids.shape[1]:], skip_special_tokens=True).strip()

# Prompt pools (kept separate so we can test generalization on held-out prompts).
TRAINING_PROMPTS = [
    "Describe the ocean.", "Tell me about Mondays.", "What is a good breakfast?",
    "Describe your favorite hobby.", "Tell me about winter.", "Describe a city street.",
    "What do you think of coffee?", "Tell me about your weekend.", "Describe a forest.",
    "What is it like to learn guitar?", "Tell me about autumn leaves.", "Describe a train journey.",
    "What do you think about gardening?", "Tell me about the night sky.", "Describe a small village.",
    "What is it like to cook dinner?", "Tell me about rainy days.", "Describe a mountain hike.",
]
HELD_OUT_PROMPTS = [
    "Tell me about libraries.", "Describe a beach.", "What do you think of tea?",
    "Tell me about spring.", "Describe a busy market.", "What is it like to ride a bike?",
]

#### From text to token ids

To read a response's log-probability we need the tokens of *prompt + response* together, plus the
index where the prompt ends — so we score **only** the response tokens, never the prompt.

In [ ]:
def build_input_ids(prompt, response):
    """Concatenate templated prompt + response into one token sequence.
    Returns (input_ids of shape [1, total_length], prompt_length)."""
    prompt_ids = tokenizer(chat_prompt_text(prompt), return_tensors="pt",
                           add_special_tokens=False).input_ids
    response_ids = tokenizer(response, return_tensors="pt",
                             add_special_tokens=False).input_ids
    input_ids = torch.cat([prompt_ids, response_ids], dim=1).to(device)
    return input_ids, prompt_ids.shape[1]

#### ★ The primitive you implement

Both algorithms call this. Fill in the gather-and-sum: convert the model's logits to per-token
log-probabilities, then pick out the log-prob of each *actual* response token (teacher forcing).

In [ ]:
def response_logps(model, input_ids, prompt_length):
    """Per-token log-probabilities of the RESPONSE tokens under `model`.
    Returns a 1-D tensor of length (number of response tokens)."""
    # SOLUTION:
    logits = model(input_ids).logits[0]                 # (total_length, vocab_size)
    token_logprobs = log_softmax(logits, dim=-1)        # (total_length, vocab_size)
    response_tokens = input_ids[0, prompt_length:]      # the actual response token ids
    # logits at positions [prompt_length-1 : -1] are the ones predicting the response tokens
    predicting_logprobs = token_logprobs[prompt_length - 1:-1]
    chosen_logprobs = predicting_logprobs.gather(-1, response_tokens[:, None]).squeeze(-1)
    return chosen_logprobs

def sequence_logp(model, prompt, response):
    """Total log-probability of a response: the sum of its per-token log-probs."""
    input_ids, prompt_length = build_input_ids(prompt, response)
    return response_logps(model, input_ids, prompt_length).sum()

#### Sanity check & visualization

Run the primitive on one short, enthusiastic answer and plot each token's log-probability. Near-zero
bars are tokens the model finds "obvious"; very negative bars are ones it finds "surprising".

In [ ]:
# Sanity check + visualization: per-token log-probs of a short, enthusiastic response.
example_prompt   = "Describe the ocean."
example_response = "I love the ocean, it is absolutely amazing!"
input_ids, prompt_length = build_input_ids(example_prompt, example_response)
with torch.no_grad():
    per_token_logprobs = response_logps(reference_model, input_ids, prompt_length).cpu().numpy()

token_strings = [tokenizer.decode([token_id]) for token_id in input_ids[0, prompt_length:].cpu().numpy()]
assert len(per_token_logprobs) == len(token_strings), "length mismatch — check the slice in response_logps"

figure, axis = plt.subplots(figsize=(10, 3.2))
axis.bar(range(len(per_token_logprobs)), per_token_logprobs, color="#1D6FB8")
axis.set_xticks(range(len(per_token_logprobs)))
axis.set_xticklabels(token_strings, rotation=60, ha="right", fontsize=8)
axis.set_ylabel("log p(token)")
axis.set_title(f"Per-token log-probabilities (reference)   |   sum = {per_token_logprobs.sum():.1f}")
plt.tight_layout(); plt.show()
print("sequence log-prob of this response under the reference:",
      f"{sequence_logp(reference_model, example_prompt, example_response).item():.2f}")

> **Note the length effect.** The total log-prob is a sum of (negative) per-token terms, so **longer
> responses get a lower total just for being longer**. This is why DPO compares the policy to a
> reference — the length bias largely cancels in the log-ratio $\log\frac{\pi_\theta}{\pi_{\mathrm{ref}}}$.

### Shared — evaluation helpers

Two small helpers reused by **both** parts: one samples many answers and returns their rewards (for a
distribution plot), the other prints reference-vs-trained answers with the reward *and its reason*.

In [ ]:
def collect_rewards(model, prompts=HELD_OUT_PROMPTS, samples_per_prompt=4, description="sampling"):
    """Sample several answers per prompt and return the list of their enthusiasm rewards."""
    rewards = []
    for prompt in tqdm(prompts, desc=description, leave=False):
        for _ in range(samples_per_prompt):
            rewards.append(reward_function(sample_response(model, prompt)))
    return rewards

def plot_reward_distributions(distributions, title):
    """distributions: list of (label, rewards, color). Overlaid histograms with means."""
    all_rewards = [reward for _, rewards, _ in distributions for reward in rewards]
    bins = np.arange(0, max(all_rewards) + 2) - 0.5
    figure, axis = plt.subplots(figsize=(7, 4))
    for label, rewards, color in distributions:
        axis.hist(rewards, bins=bins, alpha=0.55, color=color,
                  label=f"{label} (mean {np.mean(rewards):.2f})")
    axis.set_xlabel("enthusiasm reward of a sampled answer"); axis.set_ylabel("count")
    axis.legend(); axis.set_title(title)
    plt.tight_layout(); plt.show()

def print_reference_vs(model, model_label, prompts=HELD_OUT_PROMPTS[:4]):
    """Print reference vs trained-model answers with reward AND the reason for each."""
    for prompt in prompts:
        reference_answer = sample_response(reference_model, prompt)
        model_answer     = sample_response(model, prompt)
        reference_reward, reference_reasons = reward_breakdown(reference_answer)
        model_reward, model_reasons         = reward_breakdown(model_answer)
        print("=" * 92)
        print("PROMPT:", prompt)
        print(f"\n[REFERENCE]   reward = {reference_reward:.0f}")
        print("   answer:", reference_answer)
        print(f"   why: {reference_reasons['exclamation_marks']} '!'  +  positive words {reference_reasons['positive_words']}")
        print(f"\n[{model_label}]  reward = {model_reward:.0f}")
        print("   answer:", model_answer)
        print(f"   why: {model_reasons['exclamation_marks']} '!'  +  positive words {model_reasons['positive_words']}")
        print()

---
# 🟦 Part A — DPO (Direct Preference Optimization)

### Recap

DPO starts from the KL-constrained RLHF objective and a remarkable fact: its optimum has a **closed
form** (lecture section 7),

$$\pi^*(y\mid x) \;=\; \tfrac{1}{Z(x)}\,\pi_{\mathrm{ref}}(y\mid x)\,\exp\!\Big(\tfrac{1}{\beta}\,r(x,y)\Big)
\quad\Longrightarrow\quad
r(x,y) \;=\; \beta \log\frac{\pi^*(y\mid x)}{\pi_{\mathrm{ref}}(y\mid x)} + \beta\log Z(x).$$

Put that reward into Bradley–Terry; the intractable $Z(x)$ **cancels** (same prompt for $y_w$ and $y_l$),
leaving a loss that depends only on policy and reference log-probs:

$$\mathcal{L}_{\mathrm{DPO}} = -\,\mathbb{E}_{(x,\,y_w,\,y_l)}\;\log\sigma\!\Big(
\underbrace{\beta\log\tfrac{\pi_\theta(y_w\mid x)}{\pi_{\mathrm{ref}}(y_w\mid x)}}_{\hat r_\theta(x,\,y_w)}
-\underbrace{\beta\log\tfrac{\pi_\theta(y_l\mid x)}{\pi_{\mathrm{ref}}(y_l\mid x)}}_{\hat r_\theta(x,\,y_l)}\Big).$$

Its gradient **pushes the chosen response up and the rejected response down** (relative to the
reference), weighting harder mistakes more.

**DPO step = three functions:**

1. log-probs of $y_w$ and $y_l$ under $\pi_\theta$ and $\pi_{\mathrm{ref}}$ → `response_logps`
2. combine the four numbers into the loss → `dpo_loss`  *(you implement this)*
3. backpropagate and step the optimizer → `loss.backward(); optimizer.step()`

It trains **offline** on a fixed set of preference pairs — which we build next.

### A.1 — Build a preference dataset

Where do preference pairs come from? We sample several responses **from the reference model itself**,
score each with the dummy reward, and keep the highest as `chosen` and the lowest as `rejected`.

In [ ]:
def build_preference_pairs(prompts, samples_per_prompt=3):
    """For each prompt: sample a few responses, keep the most vs least enthusiastic as a pair."""
    preference_pairs = []
    for prompt in tqdm(prompts, desc="Sampling preference pairs"):
        candidates = [sample_response(reference_model, prompt) for _ in range(samples_per_prompt)]
        candidates = [candidate for candidate in candidates if candidate.strip()]
        if not candidates:
            continue
        rewards = [reward_function(candidate) for candidate in candidates]
        if max(rewards) > min(rewards):   # keep only pairs with a real preference gap
            preference_pairs.append({
                "prompt": prompt,
                "chosen": candidates[int(np.argmax(rewards))],
                "rejected": candidates[int(np.argmin(rewards))],
            })
    return preference_pairs

preference_pairs = build_preference_pairs(TRAINING_PROMPTS)
print(f"\nbuilt {len(preference_pairs)} preference pairs\n")
for pair in preference_pairs[:3]:
    print("PROMPT  :", pair["prompt"])
    print(f"  chosen   (reward={reward_function(pair['chosen']):.0f}): {pair['chosen'][:90]}")
    print(f"  rejected (reward={reward_function(pair['rejected']):.0f}): {pair['rejected'][:90]}\n")

Each kept pair shares the same prompt but has a **reward gap** — the `chosen` answer is more
enthusiastic than the `rejected` one. This fixed set is everything DPO will ever see; it never
samples again.

### A.2 ★ — Implement the DPO loss

In [ ]:
def dpo_loss(logp_policy_chosen, logp_policy_rejected,
             logp_reference_chosen, logp_reference_rejected, beta=0.1):
    """DPO loss for ONE pair, from the four sequence log-probs."""
    # SOLUTION:
    implicit_reward_chosen   = logp_policy_chosen   - logp_reference_chosen      # log pi_theta/pi_ref for y_w
    implicit_reward_rejected = logp_policy_rejected - logp_reference_rejected    # ... for y_l
    margin = implicit_reward_chosen - implicit_reward_rejected
    return -logsigmoid(beta * margin)

# toy check: chosen clearly better -> smaller loss than the reversed case
assert dpo_loss(torch.tensor(-2.0), torch.tensor(-5.0), torch.tensor(-3.0), torch.tensor(-3.0)) \
     < dpo_loss(torch.tensor(-5.0), torch.tensor(-2.0), torch.tensor(-3.0), torch.tensor(-3.0))
print("dpo_loss looks correct ✅")

The `assert` passes only when "chosen better" gives a *lower* loss than "rejected better" — a quick
check that the sign convention is right.

### A.3 — One DPO step: watch the log-probabilities move

Start the **policy** as a copy of the reference (so initially identical), measure the four log-probs,
take **one** gradient step, and measure again. The learning rate is deliberately large so a single
step is visible — real training uses many small steps.

In [ ]:
def mean_chosen_rejected_logps(model, pairs):
    """Mean sequence log-prob of chosen and of rejected responses across `pairs`."""
    chosen_logps, rejected_logps = [], []
    with torch.no_grad():
        for pair in pairs:
            chosen_logps.append(sequence_logp(model, pair["prompt"], pair["chosen"]).item())
            rejected_logps.append(sequence_logp(model, pair["prompt"], pair["rejected"]).item())
    return np.array(chosen_logps), np.array(rejected_logps)

demonstration_pairs = preference_pairs[:6]

# reference log-probs of the demo pairs (the fixed KL anchor) — compute once, as plain floats.
demo_reference_logps = {}
for pair_index, pair in enumerate(demonstration_pairs):
    demo_reference_logps[(pair_index, "chosen")]   = sequence_logp(reference_model, pair["prompt"], pair["chosen"]).item()
    demo_reference_logps[(pair_index, "rejected")] = sequence_logp(reference_model, pair["prompt"], pair["rejected"]).item()

demo_policy = load_model().train()
optimizer = torch.optim.Adam(demo_policy.parameters(), lr=1e-5)
BETA = 0.1

chosen_logp_before, rejected_logp_before = mean_chosen_rejected_logps(demo_policy, demonstration_pairs)

optimizer.zero_grad()
batch_losses = []
for pair_index, pair in enumerate(demonstration_pairs):
    ids_chosen, prompt_len_chosen     = build_input_ids(pair["prompt"], pair["chosen"])
    ids_rejected, prompt_len_rejected = build_input_ids(pair["prompt"], pair["rejected"])
    logp_policy_chosen   = response_logps(demo_policy, ids_chosen,   prompt_len_chosen).sum()
    logp_policy_rejected = response_logps(demo_policy, ids_rejected, prompt_len_rejected).sum()
    batch_losses.append(dpo_loss(
        logp_policy_chosen, logp_policy_rejected,
        demo_reference_logps[(pair_index, "chosen")], demo_reference_logps[(pair_index, "rejected")], BETA))
mean_loss = torch.stack(batch_losses).mean()
mean_loss.backward()
optimizer.step()
print(f"DPO loss this step: {mean_loss.item():.4f}   (equals log 2 = {np.log(2):.4f} when policy == reference)")

chosen_logp_after, rejected_logp_after = mean_chosen_rejected_logps(demo_policy, demonstration_pairs)
del demo_policy; gc.collect()
if device == "cuda": torch.cuda.empty_cache()

Those four numbers are easier to read as a picture — *before* vs *after*, for chosen and rejected:

In [ ]:
labels = ["chosen $y_w$", "rejected $y_l$"]
positions = np.arange(2); bar_width = 0.35
figure, axis = plt.subplots(figsize=(6.8, 4.2))
axis.bar(positions - bar_width/2, [chosen_logp_before.mean(), rejected_logp_before.mean()],
         bar_width, label="before  (= reference)", color="#B0C4DE")
axis.bar(positions + bar_width/2, [chosen_logp_after.mean(), rejected_logp_after.mean()],
         bar_width, label="after 1 DPO step", color=["#1D6FB8", "#C2185B"])
axis.annotate("", xy=(positions[0] + bar_width/2, chosen_logp_after.mean()),
              xytext=(positions[0] - bar_width/2, chosen_logp_before.mean()),
              arrowprops=dict(arrowstyle="->", color="#1D6FB8", lw=2))
axis.annotate("", xy=(positions[1] + bar_width/2, rejected_logp_after.mean()),
              xytext=(positions[1] - bar_width/2, rejected_logp_before.mean()),
              arrowprops=dict(arrowstyle="->", color="#C2185B", lw=2))
axis.set_xticks(positions); axis.set_xticklabels(labels)
axis.set_ylabel("mean sequence log-prob"); axis.legend()
axis.set_title("DPO pushes the chosen response UP and the rejected response DOWN")
plt.tight_layout(); plt.show()
print(f"chosen   : {chosen_logp_before.mean():.2f} → {chosen_logp_after.mean():.2f}   ({chosen_logp_after.mean()-chosen_logp_before.mean():+.2f})")
print(f"rejected : {rejected_logp_before.mean():.2f} → {rejected_logp_after.mean():.2f}   ({rejected_logp_after.mean()-rejected_logp_before.mean():+.2f})")

> 💡 **What to notice.** The loss starts at exactly $\log 2 \approx 0.693$ — because the policy *is* the
> reference at step 0, so the margin is $0$ and $\sigma(0)=\tfrac12$. After one step the bars separate:
> chosen up, rejected down. That contrastive push-apart is the whole of DPO — and note that
> $\pi_{\mathrm{ref}}$ never moves; it only appears inside the log-ratio.

### A.4 — Full DPO training loop

Now many small steps over all the pairs. We track the loss, the implicit-reward margin, and how the
chosen/rejected log-probs separate over epochs.

In [ ]:
dpo_policy = load_model().train()
optimizer = torch.optim.Adam(dpo_policy.parameters(), lr=1e-5)
BETA, NUM_EPOCHS = 0.1, 8

# The reference is frozen -> its log-probs never change. Compute them ONCE (with a progress bar).
reference_logp_cache = {}
for pair_index, pair in enumerate(tqdm(preference_pairs, desc="Caching reference log-probs")):
    reference_logp_cache[(pair_index, "chosen")]   = sequence_logp(reference_model, pair["prompt"], pair["chosen"]).item()
    reference_logp_cache[(pair_index, "rejected")] = sequence_logp(reference_model, pair["prompt"], pair["rejected"]).item()

loss_history, margin_history = [], []
epoch_chosen_gap, epoch_rejected_gap = [], []   # policy − reference, averaged over the demo pairs
baseline_chosen, baseline_rejected = mean_chosen_rejected_logps(reference_model, demonstration_pairs)
order = list(range(len(preference_pairs)))
for epoch in range(NUM_EPOCHS):
    random.shuffle(order)
    progress_bar = tqdm(order, desc=f"DPO epoch {epoch+1}/{NUM_EPOCHS}", leave=False)
    for pair_index in progress_bar:
        pair = preference_pairs[pair_index]
        ids_chosen, prompt_len_chosen     = build_input_ids(pair["prompt"], pair["chosen"])
        ids_rejected, prompt_len_rejected = build_input_ids(pair["prompt"], pair["rejected"])
        logp_policy_chosen   = response_logps(dpo_policy, ids_chosen,   prompt_len_chosen).sum()
        logp_policy_rejected = response_logps(dpo_policy, ids_rejected, prompt_len_rejected).sum()
        reference_chosen   = reference_logp_cache[(pair_index, "chosen")]
        reference_rejected = reference_logp_cache[(pair_index, "rejected")]
        loss = dpo_loss(logp_policy_chosen, logp_policy_rejected, reference_chosen, reference_rejected, BETA)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        loss_history.append(loss.item())
        margin_history.append(((logp_policy_chosen - reference_chosen) - (logp_policy_rejected - reference_rejected)).item())
        progress_bar.set_postfix(loss=f"{loss.item():.3f}")
    chosen_now, rejected_now = mean_chosen_rejected_logps(dpo_policy, demonstration_pairs)
    epoch_chosen_gap.append(float(chosen_now.mean()   - baseline_chosen.mean()))
    epoch_rejected_gap.append(float(rejected_now.mean() - baseline_rejected.mean()))

dpo_policy.eval()
print("DPO training done.")

Now let's plot what happened across **all** the training steps:

In [ ]:
figure, axes = plt.subplots(1, 3, figsize=(14, 3.6))
axes[0].plot(loss_history, color="#1D6FB8"); axes[0].set_title("DPO loss"); axes[0].set_xlabel("step"); axes[0].grid(alpha=0.3)
axes[1].plot(margin_history, color="#7E3C9E"); axes[1].axhline(0, color="black", linewidth=0.6)
axes[1].set_title("implicit-reward margin (chosen − rejected)"); axes[1].set_xlabel("step"); axes[1].grid(alpha=0.3)
axes[2].plot(range(1, NUM_EPOCHS+1), epoch_chosen_gap, "o-", color="#1D6FB8", label="chosen")
axes[2].plot(range(1, NUM_EPOCHS+1), epoch_rejected_gap, "o-", color="#C2185B", label="rejected")
axes[2].axhline(0, color="black", linewidth=0.6)
axes[2].set_title("log-prob vs reference, over epochs"); axes[2].set_xlabel("epoch"); axes[2].legend(); axes[2].grid(alpha=0.3)
plt.tight_layout(); plt.show()

**Reading the curves.** *Left* — the loss falls as the policy learns to prefer the chosen answers.
*Middle* — the implicit-reward margin (chosen − rejected, relative to the reference) grows positive.
*Right* — across epochs the chosen log-prob rises above the reference while the rejected sinks below:
the single-step push-apart, accumulated.

#### 🎞️ Animation — the chosen/rejected gap opening up

As training proceeds, the chosen bar rises above the reference and the rejected bar sinks below it.

In [ ]:
from matplotlib import animation
from IPython.display import HTML

figure, axis = plt.subplots(figsize=(5, 4))
y_low  = min(epoch_rejected_gap) - 1
y_high = max(epoch_chosen_gap) + 1

def draw_frame(epoch_index):
    axis.clear()
    axis.bar(["chosen $y_w$", "rejected $y_l$"],
             [epoch_chosen_gap[epoch_index], epoch_rejected_gap[epoch_index]],
             color=["#1D6FB8", "#C2185B"])
    axis.axhline(0, color="black", linewidth=0.8)
    axis.set_ylim(y_low, y_high)
    axis.set_ylabel("log-prob relative to reference")
    axis.set_title(f"after epoch {epoch_index+1}/{NUM_EPOCHS}")

gap_animation = animation.FuncAnimation(figure, draw_frame, frames=NUM_EPOCHS, interval=600)
plt.close(figure)
HTML(gap_animation.to_jshtml())

### A.5 — Did the output distribution shift?

In [ ]:
reference_rewards = collect_rewards(reference_model, description="reference")
dpo_rewards       = collect_rewards(dpo_policy,       description="DPO policy")
plot_reward_distributions(
    [("reference", reference_rewards, "#6B6A65"), ("DPO policy", dpo_rewards, "#1D6FB8")],
    title="DPO shifted the output distribution toward the reward")
print(f"mean enthusiasm:  reference {np.mean(reference_rewards):.2f}  →  DPO {np.mean(dpo_rewards):.2f}")

Numbers are one thing — let's read the actual answers, each with its reward and the exact tokens that
earned it:

In [ ]:
print_reference_vs(dpo_policy, "DPO POLICY")

> 💡 **Takeaway.** The DPO histogram sits to the right of the reference. The model was never *told* to
> use exclamation marks — it learned the behavior purely from **preferences**. If you push training much
> further, answers degrade into `!!!` spam: over-optimization / Goodhart (lecture section 9).

---
# 🟧 Part B — REINFORCE (policy gradient)

### Recap

The policy gradient theorem (lecture section 6):

$$\nabla_\theta J(\theta) = \mathbb{E}_{\tau\sim\pi_\theta}\!\Big[\textstyle\sum_t \nabla_\theta\log\pi_\theta(a_t\mid s_t)\;\Psi_t\Big].$$

REINFORCE's one choice is to use the **return-to-go** as the credit signal $\Psi_t$:

$$\Psi_t = G_t = \sum_{t'=t}^{T}\gamma^{\,t'-t}\,r_{t'}.$$

#### ❓ Why is there no reward-to-go term in our code?

In the **LLM setting the reward is terminal**: the reward model scores the *whole* completion once, at
the end. There are no intermediate rewards $r_t$ — only a single $R(x,y)$ at the final step. With
$\gamma = 1$, the return-to-go from *every* token position is therefore the **same** number:

$$G_t = \underbrace{0 + 0 + \dots + 0}_{\text{no intermediate rewards}} + R(x,y) = R(x,y)\quad\text{for all } t.$$

So there is nothing per-token to accumulate — "reward-to-go" collapses to the constant $R(x,y)$.
Substituting $G_t = R$ into the theorem, the constant pulls out of the sum:

$$\nabla_\theta J = \mathbb{E}\Big[R(x,y)\,\textstyle\sum_t \nabla_\theta\log\pi_\theta(y_t\mid x,y_{<t})\Big]
= \mathbb{E}\big[R(x,y)\,\nabla_\theta \log\pi_\theta(y\mid x)\big].$$

**We simply scale the whole-sequence log-prob by its reward.** To cut variance we subtract a
**baseline** $b$, turning the reward into an *advantage* $R-b$ — exactly the idea behind PPO's value
baseline and GRPO's group mean.

In [ ]:
# Figure: in an LLM episode the reward arrives ONCE, at the end -> G_t = R for every token.
example_tokens = ["The", "movie", "was", "absolutely", "great", "!"]
num_tokens = len(example_tokens)
figure, axis = plt.subplots(figsize=(11, 2.8))
for position, token in enumerate(example_tokens):
    is_last = position == num_tokens - 1
    axis.add_patch(plt.Rectangle((position, 0), 0.92, 1, facecolor="#FCEFE0", edgecolor="#B5730F"))
    axis.text(position + 0.46, 0.5, token, ha="center", va="center", fontsize=11)
    immediate = "R(x,y)" if is_last else "0"
    axis.text(position + 0.46, -0.55, f"$r_{position+1}={immediate}$", ha="center", va="center",
              fontsize=9, color="#B5730F")
    axis.text(position + 0.46, -1.15, "$G=R$", ha="center", va="center", fontsize=9, color="#2E7D46")
axis.annotate("reward model scores\nthe whole sequence", xy=(num_tokens - 0.1, 0.5),
              xytext=(num_tokens + 0.2, 0.5), va="center", fontsize=9, color="#B5730F",
              arrowprops=dict(arrowstyle="->", color="#B5730F"))
axis.set_xlim(-0.3, num_tokens + 2.2); axis.set_ylim(-1.6, 1.4); axis.axis("off")
axis.set_title("Terminal reward: every token's return-to-go equals the single sequence reward R")
plt.tight_layout(); plt.show()

Read it left to right: the reward model stays silent until the sequence ends, then returns a single
score `R`. Every token therefore shares the *same* credit signal — which is why the code multiplies
the whole-sequence log-prob by one reward instead of summing per-step returns.

**REINFORCE step = these functions:**

1. sample completions from the current policy → `sample_response`
2. score each with the reward model → `reward_function`
3. baseline = mean reward of the batch (our advantage $R-b$)
4. log-prob of each sample under $\pi_\theta$ → `sequence_logp`
5. loss $= -(R-b)\,\log\pi_\theta(y\mid x)$ → `reinforce_loss`  *(you implement this)*
6. backpropagate and step the optimizer

Unlike DPO it needs **no reference model and no pairs** — just samples and their scalar rewards, freshly
generated each step (**online**).

### B.1 ★ — Implement the REINFORCE loss

In [ ]:
def reinforce_loss(sequence_log_prob, reward, baseline=0.0):
    """REINFORCE loss for ONE sample. `sequence_log_prob` is log pi_theta(y|x) (a tensor with grad)."""
    # SOLUTION:
    advantage = reward - baseline
    return -advantage * sequence_log_prob

# toy check: with a positive advantage, increasing log-prob must DECREASE the loss
log_prob_variable = torch.tensor(-5.0, requires_grad=True)
reinforce_loss(log_prob_variable, reward=1.0, baseline=0.0).backward()
assert log_prob_variable.grad.item() < 0
print("reinforce_loss looks correct ✅")

Same idea as the DPO check: a positive advantage must *lower* the loss by *raising* that sample's
log-probability.

### B.2 — One REINFORCE step: watch the log-probabilities move

Sample several completions to one prompt, score them, set the baseline to the **mean reward**, and take
one step. REINFORCE raises the log-prob of **above-average** samples and lowers **below-average** ones —
in proportion to the advantage $R-b$.

In [ ]:
reinforce_demo_policy = load_model().train()
optimizer = torch.optim.Adam(reinforce_demo_policy.parameters(), lr=1e-5)

reinforce_prompt = "Tell me about Mondays."
sampled_responses = [sample_response(reinforce_demo_policy, reinforce_prompt, temperature=1.1) for _ in range(8)]
sampled_responses = [response for response in sampled_responses if response.strip()]
sample_rewards = np.array([reward_function(response) for response in sampled_responses])
baseline = sample_rewards.mean()

log_prob_before = np.array([sequence_logp(reinforce_demo_policy, reinforce_prompt, response).item()
                            for response in sampled_responses])

optimizer.zero_grad()
step_losses = [reinforce_loss(sequence_logp(reinforce_demo_policy, reinforce_prompt, response),
                              float(reward), float(baseline))
               for response, reward in zip(sampled_responses, sample_rewards)]
torch.stack(step_losses).mean().backward()
optimizer.step()
print(f"baseline (mean reward of the batch) = {baseline:.2f}")

log_prob_after = np.array([sequence_logp(reinforce_demo_policy, reinforce_prompt, response).item()
                           for response in sampled_responses])
del reinforce_demo_policy; gc.collect()
if device == "cuda": torch.cuda.empty_cache()

Plotting each sample's change in log-probability against its advantage shows the rule directly:

In [ ]:
advantages = sample_rewards - baseline
figure, axis = plt.subplots(figsize=(6.8, 4.2))
point_colors = ["#2E7D46" if advantage > 0 else ("#C0392B" if advantage < 0 else "#6B6A65")
                for advantage in advantages]
axis.scatter(advantages, log_prob_after - log_prob_before, c=point_colors, s=70)
axis.axhline(0, color="black", linewidth=0.6); axis.axvline(0, color="black", linewidth=0.6)
axis.set_xlabel("advantage  (reward − baseline)")
axis.set_ylabel("Δ log π(sample)  after one step")
axis.set_title("REINFORCE lifts above-average samples, lowers below-average ones")
axis.text(0.02, 0.92, "push UP (A>0)", transform=axis.transAxes, color="#2E7D46", fontsize=9)
axis.text(0.02, 0.05, "push DOWN (A<0)", transform=axis.transAxes, color="#C0392B", fontsize=9)
plt.tight_layout(); plt.show()

for response, reward, delta in sorted(zip(sampled_responses, sample_rewards, log_prob_after - log_prob_before),
                                      key=lambda item: -item[1])[:4]:
    print(f"reward={reward:.0f}  Δlogp={delta:+.2f}  | {response[:70]}")

> 💡 **DPO vs REINFORCE — the contrast.** DPO needs **pairs + a reference** and moves chosen vs rejected
> apart; REINFORCE needs **samples + a scalar reward** and moves each sample by its advantage, with no
> reference at all. With only 8 noisy samples the scatter will not be perfectly clean — a reminder that
> REINFORCE is **high-variance**, which is exactly why PPO/GRPO add baselines, clipping, and a KL term.

### B.3 — Full REINFORCE training loop

Now the **online** loop: at each step we generate a fresh **group** of completions for a prompt, score
them, use the group's **mean reward as the baseline** (so the advantage is $r_i-\bar r$), and update.
This is REINFORCE with a group baseline — the same idea GRPO uses (GRPO additionally divides by the
group's standard deviation; see the note below).

> ⏳ This is the most compute-heavy cell (it generates `GROUP_SIZE` samples every step). On a **T4** it
> takes a couple of minutes; on CPU, reduce `NUM_PASSES`, `GROUP_SIZE`, or the prompt list.

In [ ]:
reinforce_policy = load_model().train()
optimizer = torch.optim.Adam(reinforce_policy.parameters(), lr=1e-4)

REINFORCE_PROMPTS = list(TRAINING_PROMPTS[:8])   # a copy — we shuffle it
GROUP_SIZE = 6                                    # completions sampled per step
SAMPLING_TEMPERATURE = 1.1                        # a bit hot, so groups vary (needed for signal)
NUM_PASSES = 6                                    # passes over the prompt list

mean_reward_history = []
total_steps = NUM_PASSES * len(REINFORCE_PROMPTS)
progress_bar = tqdm(total=total_steps, desc="REINFORCE training")
for pass_index in range(NUM_PASSES):
    random.shuffle(REINFORCE_PROMPTS)
    for prompt in REINFORCE_PROMPTS:
        # 1) sample a group of completions; 2) score them
        completions = [sample_response(reinforce_policy, prompt, temperature=SAMPLING_TEMPERATURE)
                       for _ in range(GROUP_SIZE)]
        completions = [completion for completion in completions if completion.strip()]
        rewards = np.array([reward_function(completion) for completion in completions])
        mean_reward_history.append(float(rewards.mean()))
        # 3) baseline = group mean. If the whole group ties, there is no signal -> skip the update.
        baseline = float(rewards.mean())
        if rewards.std() > 1e-6:
            optimizer.zero_grad()
            step_losses = [reinforce_loss(sequence_logp(reinforce_policy, prompt, completion),
                                          float(reward), baseline)
                           for completion, reward in zip(completions, rewards)]
            torch.stack(step_losses).mean().backward()
            optimizer.step()
        progress_bar.update(1); progress_bar.set_postfix(mean_reward=f"{rewards.mean():.2f}")
progress_bar.close()
reinforce_policy.eval()
print("REINFORCE training done.")

Let's watch the reward the policy earns as training proceeds:

In [ ]:
# Learning curve: mean group reward per step (noisy) with a running average (REINFORCE = high variance).
mean_reward_array = np.array(mean_reward_history)
window = 5
running_average = np.convolve(mean_reward_array, np.ones(window)/window, mode="valid")
figure, axis = plt.subplots(figsize=(8, 4))
axis.plot(mean_reward_array, color="#B5730F", alpha=0.4, label="mean group reward (per step)")
axis.plot(range(window-1, len(mean_reward_array)), running_average, color="#B5730F", lw=2,
          label=f"running average (window {window})")
axis.set_xlabel("step"); axis.set_ylabel("mean reward of the sampled group")
axis.set_title("REINFORCE learning curve — reward rises (noisily) as the policy improves")
axis.legend(); axis.grid(alpha=0.3); plt.tight_layout(); plt.show()

**Reading the curve.** Unlike DPO's smooth descent, this is a *noisy* climb — each point is a fresh,
random group of samples, so the variance is high. The running average reveals the upward trend; flat
stretches are groups that happened to tie (no signal, so the update is skipped).

### B.4 — Did the output distribution shift?

In [ ]:
# Reuse the reference distribution from Part A; sample the REINFORCE policy on the same held-out prompts.
reinforce_rewards = collect_rewards(reinforce_policy, description="REINFORCE policy")
plot_reward_distributions(
    [("reference", reference_rewards, "#6B6A65"), ("REINFORCE policy", reinforce_rewards, "#B5730F")],
    title="REINFORCE shifted the output distribution toward the reward")
print(f"mean enthusiasm:  reference {np.mean(reference_rewards):.2f}  →  REINFORCE {np.mean(reinforce_rewards):.2f}")

And the answers themselves, with reward and reason:

In [ ]:
print_reference_vs(reinforce_policy, "REINFORCE")

> 💡 **Takeaway.** REINFORCE is **noisier** than DPO and slower to take off (sparse reward: many groups
> tie at 0 early, giving no gradient — the cold-start problem). If the curve looks flat, raise
> `GROUP_SIZE`, `SAMPLING_TEMPERATURE`, `NUM_PASSES`, or the learning rate. Train much longer and it
> reward-hacks into `!!!` spam — Goodhart again, and the reason GRPO keeps a KL term.

---
# Part C — DPO vs REINFORCE, side by side

Both methods optimized the **same** toy reward; let's overlay all three output distributions.

In [ ]:
plot_reward_distributions(
    [("reference",        reference_rewards, "#6B6A65"),
     ("DPO policy",       dpo_rewards,       "#1D6FB8"),
     ("REINFORCE policy", reinforce_rewards, "#B5730F")],
    title="Reference vs DPO vs REINFORCE — enthusiasm reward of sampled answers")
print(f"mean enthusiasm   reference {np.mean(reference_rewards):.2f}   "
      f"DPO {np.mean(dpo_rewards):.2f}   REINFORCE {np.mean(reinforce_rewards):.2f}")

Both trained policies sit to the right of the reference. On this toy setup DPO usually gives the
cleaner, larger shift (it learns from curated pairs), while REINFORCE gets there more noisily, by
trial and error — the offline-vs-online tradeoff in miniature.

### Recap

| | DPO (🟦) | REINFORCE (🟧) |
|---|---|---|
| Needs | preference **pairs** + a **reference** model | **samples** + a scalar **reward** |
| Data | **offline**, fixed dataset | **online**, freshly sampled each step |
| Update | push chosen up, rejected down (contrastive) | scale each sample's log-prob by its advantage $R-b$ |
| Reward-to-go | not used | collapses to terminal $R$ (scale the whole-sequence log-prob) |
| Variance | low (deterministic targets) | high (Monte-Carlo samples) → baselines, GRPO, PPO |
| You wrote | `dpo_loss` | `reinforce_loss` |

Both reuse the same primitive `response_logps` → $\log\pi(y\mid x)$, and both shift the output
distribution toward the reward — DPO smoothly and cheaply from fixed pairs, REINFORCE noisily from
fresh on-policy samples.

**Extensions:** (1) add GRPO-style std-normalization to the REINFORCE advantage and compare stability;
(2) sweep `BETA` (DPO) / `GROUP_SIZE` (REINFORCE); (3) over-train either and watch the `!!!` spam appear
(Goodhart); (4) swap `reward_function` for a different toy preference (brevity, politeness, a keyword).